In [2]:

from pathlib import Path
from decouple import AutoConfig

import numpy as np
from pinecone import Pinecone

In [5]:
cwd = Path.cwd()
project_dir = cwd / "langchain" if (cwd / "langchain").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
api_key = config("PINECONE_API_KEY")

In [7]:
pinecone_client = Pinecone(api_key=api_key)

Pinecone

In [11]:
pinecone_client.create_index(
    name="nlp-vetores",
    dimension=2048,
    metric="cosine",
)

Name:,nlp-vetores
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://nlp-vetores-q5qe5xi.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [12]:
indices = pinecone_client.list_indexes()
for index in indices:
    index_name = index["name"]
    print(f"Index {index_name}")
    print(pinecone_client.describe_index(index_name))

Index nlp-vetores
IndexModel(name='nlp-vetores', status='Ready', host='https://nlp-vetores-q5qe5xi.svc.aped-4627-b74a.pinecone.io', deployment='Managed', deletion_protection='disabled', schema_fields=2)


In [13]:
indice_nome = 'nlp-vetores'

In [ ]:
vetores = [np.random.normal(0, 1, 2048).tolist() for _ in range(5)]
ids = ['a', 'b', 'c', 'd', 'e']
indice = pinecone_client.Index(indice_nome)
indice.upsert(vectors=list(zip(ids, vetores)))

UpsertResponse(upserted_count=5)

In [15]:
print(len(vetores))
print(vetores[4])

5
[0.1877507778089369, -0.36534238287730825, -0.6361107201224446, 0.4201781915407554, 1.0396881670094607, -0.40206443361363636, -0.36691258684609557, 0.4426016182073549, 0.31747642909033336, -1.0936491894649802, -1.010753351864312, 1.1886780901981164, -1.066792147827133, 0.7791998840682952, -0.525781918697782, -1.0720977018832252, -1.3890771998137685, 0.3085388235594563, -0.09978736788607609, 1.0641246705397178, 0.8873680500243256, -0.18968729002078663, -0.4876000438546453, 0.4083408728947119, -0.3721760329779057, -2.0928637336578553, -0.4416289162188672, -1.9874233900208533, 0.5228064787111891, -0.015440193796053645, 1.989570498498781, -1.4014858161830666, 0.5281916741395842, -0.023060412822389294, -0.7386467412688601, 1.272198139675012, -0.9531008127611993, 1.3499206900700684, 1.3022590331532324, -0.78584914039901, 0.45073680347075434, -0.4406888164033697, -1.9339797382419885, -1.1387439301285898, -2.636938955939825, 0.2109853524250617, -0.7170230390284722, 1.096358883959937, -0.0147

In [16]:
print(indice.fetch(ids=['c']))

FetchResponse(vectors={'c': Vector(id='c', values=[1.39519739, -1.51957953, -0.747206509, ...2045 more], sparse_values=None)}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 09:56:42 GMT', 'content-type': 'application/json', 'content-length': '24835', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '41', 'x-envoy-upstream-service-time': '41', 'x-pinecone-response-duration-ms': '42', 'grpc-status': '0', 'server': 'envoy'}))


In [17]:
response = indice.fetch(ids=['c'])
if 'vectors' in response and 'c' in response['vectors']:
    retorna_valor = response['vectors']['c']['values']
    atualiza_vetor = [x + 1 for x in retorna_valor]
    indice.upsert(vectors=[('c', atualiza_vetor)])
    print(indice.fetch(ids=['c']))
else:
    print("Vetor com ID 'c' nāo encontrado no índice.")

FetchResponse(vectors={'c': Vector(id='c', values=[1.39519739, -1.51957953, -0.747206509, ...2045 more], sparse_values=None)}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 10:02:22 GMT', 'content-type': 'application/json', 'content-length': '24835', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '38', 'x-envoy-upstream-service-time': '38', 'x-pinecone-response-duration-ms': '39', 'grpc-status': '0', 'server': 'envoy'}))


In [18]:
indice.delete(ids=['d', 'e'])

In [19]:
print(indice.fetch(ids=['d', 'e']))

FetchResponse(vectors={}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 10:03:06 GMT', 'content-type': 'application/json', 'content-length': '69', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '2', 'x-envoy-upstream-service-time': '3', 'x-pinecone-response-duration-ms': '4', 'grpc-status': '0', 'server': 'envoy'}))


In [20]:
print(indice.describe_index_stats())

DescribeIndexStatsResponse(dimension=2048, total_vector_count=3, metric='cosine', namespaces=1)


In [21]:
indice.upsert(
    vectors=list(zip(ids, vetores)), 
    namespace='namespace1'
)
indice.upsert(
    vectors=list(
        zip(
            ['x', 'y', 'z'],
            [np.random.normal(0, 1, 2048).tolist() for _ in range(3)]
        )
    ),
    namespace="namespace2"
)

UpsertResponse(upserted_count=3)

In [22]:
print(indice.fetch(ids=['a'], namespace='namespace1'))

FetchResponse(vectors={'a': Vector(id='a', values=[-1.34790528, 0.705231488, 0.35309574, ...2045 more], sparse_values=None)}, namespace='namespace1', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 10:12:46 GMT', 'content-type': 'application/json', 'content-length': '24806', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '37', 'x-envoy-upstream-service-time': '38', 'x-pinecone-response-duration-ms': '39', 'grpc-status': '0', 'server': 'envoy'}))


In [23]:
print(indice.fetch(ids=['x'], namespace='namespace2'))

FetchResponse(vectors={'x': Vector(id='x', values=[0.655037344, 0.190008491, -2.0784502, ...2045 more], sparse_values=None)}, namespace='namespace2', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 10:13:01 GMT', 'content-type': 'application/json', 'content-length': '24797', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '45', 'x-envoy-upstream-service-time': '46', 'x-pinecone-response-duration-ms': '46', 'grpc-status': '0', 'server': 'envoy'}))


In [26]:
indice.delete(ids=['x'], namespace='namespace2')
print(indice.fetch(ids=['x'], namespace='namespace2'))

FetchResponse(vectors={}, namespace='namespace2', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 10:14:27 GMT', 'content-type': 'application/json', 'content-length': '80', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '52', 'x-envoy-upstream-service-time': '53', 'x-pinecone-response-duration-ms': '54', 'grpc-status': '0', 'server': 'envoy'}))


In [27]:
query_vector = np.random.normal(0, 1, 2048).tolist()
print(indice.query(vector=query_vector, top_k=3, include_values=False))

QueryResponse(matches=[ScoredVector(id='b', score=0.0166161694, values=[]), ScoredVector(id='a', score=-0.00587364146, values=[]), ScoredVector(id='c', score=-0.00976145919, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Fri, 06 Nov 2026 10:15:26 GMT', 'content-type': 'application/json', 'content-length': '218', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '3', 'x-pinecone-request-latency-ms': '195', 'x-envoy-upstream-service-time': '73', 'x-pinecone-response-duration-ms': '197', 'grpc-status': '0', 'server': 'envoy'}))
